In [1]:
import pandas as pd
import numpy as np

In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

In [3]:
df= pd.read_csv(url)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [5]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [6]:
base = ['engine_displacement','horsepower','vehicle_weight']

In [7]:
df.horsepower.median()

np.float64(254.0)

In [8]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [9]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [10]:
def rmse(y, y_pred):
    error = y - y_pred
    se = error ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [11]:
df_train1 = df_train.fillna(0, inplace = False).drop(columns ='fuel_efficiency_mpg')
df_val1 =df_val.fillna(0, inplace = False).drop(columns = 'fuel_efficiency_mpg')
df_test1 = df_test.fillna(0, inplace = False).drop(columns = 'fuel_efficiency_mpg')

In [12]:
y1 = df_train['fuel_efficiency_mpg'].values
X1 = df_train1[base].values

In [13]:
w0, w =train_linear_regression(X1,y1)

In [14]:
w0, w = train_linear_regression(X1,y1)
y_pred1 = w0 + X1.dot(w)
round(rmse(y1,y_pred1),3)

np.float64(2.592)

In [15]:
df_train2 = df_train.fillna(df_train.mean(numeric_only=True), inplace = False).drop(columns ='fuel_efficiency_mpg')
df_val2 =df_val.fillna(df_val.mean(numeric_only=True), inplace = False).drop(columns = 'fuel_efficiency_mpg')
df_test2 = df_test.fillna(df_test.mean(numeric_only=True), inplace = False).drop(columns = 'fuel_efficiency_mpg')

In [16]:
y2 = df_train['fuel_efficiency_mpg'].values
X2= df_train2[base].values

In [17]:
w0, w = train_linear_regression(X2,y2)
y_pred = w0 + X2.dot(w)

In [18]:
print (y_pred)

[32.43077377 30.23059769 30.34535155 ... 30.3602491  29.78987006
 29.95933313]


In [19]:
round(rmse(y2, y_pred),3)

np.float64(2.635)

QUESTION 4

In [20]:
def train_linear_regression_reg(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [21]:
df_train1 = df_train.fillna(0, inplace = False).drop(columns ='fuel_efficiency_mpg')
df_val1 =df_val.fillna(0, inplace = False).drop(columns = 'fuel_efficiency_mpg')
df_test1 = df_test.fillna(0, inplace = False).drop(columns = 'fuel_efficiency_mpg')
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

In [22]:
df_train1 = df_train.fillna(0, inplace = False).drop(columns ='fuel_efficiency_mpg')
y_train = df_train['fuel_efficiency_mpg'].values

In [23]:
def train_linear_regression_reg(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    
    # Do not regularize the intercept (index 0)
    reg_matrix = np.eye(XTX.shape[0])
    reg_matrix[0, 0] = 0
    XTX = XTX + r * reg_matrix

    # Use solve instead of inv for numerical stability
    w_full = np.linalg.solve(XTX, X.T.dot(y))
    
    return w_full[0], w_full[1:]

In [24]:
def prepare_X(df):
    df = df.copy()
    features = base.copy()

    df['age'] = 2026 - df.model_year
    features.append('age')

    for c, values in categories.items():
        for v in values:
            df['%s_%s' % (c, v)] = (df[c] == v).astype('int')
            features.append('%s_%s' % (c, v))

    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

In [25]:
categorical_features = ['age']
categories = {}
for c in categorical_features:
    categories[c] = list(df[c].value_counts().head(5).index)

KeyError: 'age'

In [ ]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [ ]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare_X(df_train)
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)

    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred).round(4)

    print(r, w0, score)

0 3.953656778807213e+17 599.4264
0.01 24.35698351010121 1.7493
0.1 24.204093099856905 1.7487
1 22.952607543198155 1.7492
5 19.30572698165699 1.7899
10 16.343098525605274 1.8661
100 4.801923342109909 2.4935


Question 5

In [ ]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

In [ ]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_scores = []
for s in seeds:
    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    df_train1 = df_train.fillna(0, inplace = False).drop(columns ='fuel_efficiency_mpg').values
    df_val1 =df_val.fillna(0, inplace = False).drop(columns = 'fuel_efficiency_mpg').values
    df_test1 = df_test.fillna(0, inplace = False).drop(columns = 'fuel_efficiency_mpg').values
    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values
    y_test = df_test['fuel_efficiency_mpg'].values

   
    w0, w = train_linear_regression(df_train1, y_train)
    y_pred = w0 + df_val1.dot(w)
    score = rmse(y_val, y_pred)
    
    
    rmse_scores.append(score)
    print(f"Seed {s}: RMSE = {score:.3f}")


print(f"\nAverage RMSE: {np.mean(rmse_scores):.3f} ± {np.std(rmse_scores):.3f}")